<a href="https://colab.research.google.com/github/areebaeman234-ux/ML-Internship/blob/main/Copy_of_capstone.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Capstone — mirrors your deployed research paper

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

# Ranking Signals and Page-Level CTR: An Observational Analysis Using Search Performance Data

## Abstract

This study investigates which available page-level search performance signals are associated with observed click-through rate (CTR). Using the March 2026 FlyRank ML Internship dataset, I defined CTR as Google Search Console clicks divided by Google Search Console impressions and evaluated a small set of available performance signals. I compared a Random Forest regression model with a mean-CTR baseline and then repeated the evaluation using a client-grouped 80/20 split to reduce the risk of overly optimistic results from shared clients. On the client-grouped split, the Random Forest achieved an R² of -0.0370 and an MAE of 0.001392, compared with -0.0001 R² and 0.001410 MAE for the baseline, so the model did not outperform the baseline on R². These results provide directional decision-support evidence about the usefulness of the tested signals, but they do not establish causal relationships or demonstrate that the model can reliably predict future page performance.


## 1. Question
## 1. Introduction / Problem Statement

Search performance can vary across pages, and teams need practical ways to identify which available signals are useful for prioritization. Click-through rate (CTR) is one measurable page-level performance outcome because it relates clicks to search impressions.

The goal of this project was to investigate whether available ranking and search-performance signals were associated with observed page-level CTR. The analysis focused on measurable signals rather than attempting to explain or reproduce the search engine's ranking system.

The research question was:

**Which available page-level search performance signals are associated with CTR, and can they support useful prioritization decisions?**

The decision goal was to determine whether a simple machine-learning model provided more useful predictive information than a basic baseline. The analysis therefore compared a Random Forest regression model against a mean-CTR baseline and then checked the result using a stricter client-grouped validation design.

This work is observational. The results describe patterns measured in the available data and should not be interpreted as evidence that any individual signal causes changes in CTR.


## 2. Data


### 2.1 Dataset and Release

The analysis used the **March 2026 development release** of the FlyRank ML Internship dataset. The data was accessed from the FlyRank internship warehouse using DuckDB and queried from Parquet files.

The broader warehouse contains production search-performance data. For development and analysis, I used the March 2026 release rather than the sealed June 2026 testing release.

The analysis was conducted at the page-level performance row used in the dataset. Client and content identifiers were used only where necessary for grouping and validation and were not exposed in the public results.

### 2.2 Date Window

The development analysis was restricted to data from **March 2026**. Future data was not used for feature construction or model training.

The June 2026 sealed release was not used during development because it is intended for testing rather than model development.

### 2.3 Target Variable

The target variable was Google Search Console click-through rate:

**CTR = GSC clicks / GSC impressions**

Rows with zero impressions cannot produce a meaningful CTR value and were therefore handled separately rather than being treated as ordinary CTR observations.

### 2.4 Features

The final Random Forest model used:

* `gsc_avg_position`
* `gsc_impressions`

The model did not use `gsc_clicks` as an input feature because clicks are part of the CTR calculation. Including clicks as a feature would allow information from the target to enter the model and create a leakage risk.

### 2.5 Data Exclusions

Rows that could not provide a valid CTR denominator were excluded from the modeling dataset. Public outputs also exclude client names, URLs, private search queries, and other identifying information.

These exclusions were made to keep the target definition valid and the published analysis public-safe.


## 3. Methodology

### 3.1 Analytical Approach

The analysis followed a simple progression:

1. Define the research question and target.
2. Audit the available signals.
3. Create a rule-based baseline.
4. Train a Random Forest regression model.
5. Compare the model with the baseline on the same test set.
6. Repeat the evaluation using client-grouped validation.
7. Audit possible leakage and interpret the results conservatively.

The purpose was not to maximize model complexity. The purpose was to determine whether the tested signals provided predictive information beyond a simple baseline.


### 3.2 Target Definition

The prediction target was page-level Google Search Console CTR.

CTR was calculated as:

**CTR = gsc_clicks / gsc_impressions**

The target is continuous, so the task was treated as a regression problem.

The observed target distribution was highly sparse. In the modeling data, the mean CTR was approximately **0.000640**, while the median and 75th percentile were both **0**, indicating that many observations had zero observed CTR.
### 3.3 Features

The final model used two input features:

| Feature            | Description                      | Reason for use                                                                 |
| ------------------ | -------------------------------- | ------------------------------------------------------------------------------ |
| `gsc_avg_position` | Observed average search position | Represents search-position information available before calculating the target |
| `gsc_impressions`  | Observed search impressions      | Represents the amount of observed search exposure                              |

`gsc_clicks` was deliberately excluded from the feature set because clicks are used directly to calculate CTR. Using clicks as an input would create target leakage.

### 3.4 Baseline

The baseline predicted the mean CTR of the training data for every observation in the test set.

This provides a simple reference point. A useful model should perform better than this baseline on the same held-out observations.

The baseline was evaluated using the same test observations as the Random Forest so that the comparison was fair.

### 3.5 Machine-Learning Model

The first machine-learning model was a Random Forest regression model.

Random Forest was selected as a simple non-linear model that can capture relationships between numeric features without requiring a specific linear relationship between the features and CTR.

The model was trained only on the training portion of the data. Predictions were then generated for the held-out test set.

Model performance was measured using:

* **R²** — measures how much variation is explained relative to the baseline mean prediction.
* **MAE** — measures the average absolute prediction error.
### 3.6 Validation Design

The initial Week-5 experiment used a random 80/20 train-test split. However, observations from the same client could appear in both training and test data, which could make the evaluation less representative of performance on unseen clients.

To address this concern, the validation audit used a **client-grouped 80/20 split**. Entire clients were assigned to either the training or test set, preventing the same client from appearing in both groups.

The final grouped evaluation contained:

* **7,920 training rows**
* **1,980 test rows**
* **44 training clients**
* **11 test clients**
* **0 client overlap**

This provides a stricter estimate of how the model performs when evaluated on clients not seen during training.

### 3.7 Leakage Checks

Several checks were performed before interpreting the model results.

First, the target was calculated from `gsc_clicks` and `gsc_impressions`, while `gsc_clicks` was excluded from the model features.

Second, the validation split was grouped by `client_hash_id`, resulting in zero client overlap between training and test data.

Third, the baseline and Random Forest were evaluated on the same held-out test observations.

Finally, the analysis used the March 2026 development data and did not use the sealed June 2026 testing release during development.

These checks reduce important sources of leakage, although they do not guarantee that all possible sources of bias are absent.


## 4. Results (vs baseline)

### 4.1 Model vs Baseline

The final comparison used the client-grouped 80/20 validation split. Both methods were evaluated on the same 1,980 held-out observations.

| Method                |      R² |      MAE |
| --------------------- | ------: | -------: |
| Grouped baseline      | -0.0001 | 0.001410 |
| Grouped Random Forest | -0.0370 | 0.001392 |

The Random Forest had a slightly lower MAE than the baseline, but its R² was lower. Therefore, the Random Forest did **not** beat the baseline overall on the grouped validation design.

The result is important because the stricter validation design produced a different picture from the earlier random split. This indicates that the earlier model performance may have benefited from having observations from the same clients represented across training and testing.

The appropriate conclusion is therefore that the tested features did not provide sufficient evidence of improved generalization beyond the simple baseline under client-grouped validation.


## 5. Limitations


This analysis has several limitations.

First, the observed CTR distribution is highly sparse, with many observations having zero observed CTR. This makes CTR difficult to model reliably using a small set of signals.

Second, the final model used only two features: average position and impressions. Additional features may contain useful information that was not tested in this analysis.

Third, the Random Forest did not outperform the mean-CTR baseline on the client-grouped validation split. Therefore, the model should not be presented as a production-ready CTR prediction system.

Fourth, the analysis is observational. Associations between signals and CTR do not demonstrate that changing a signal would cause CTR to change.

Finally, the validation design improves the assessment of generalization across clients, but it does not eliminate all possible sources of sampling bias or distribution shift.

The findings should therefore be interpreted as **observed, measured, and directional evidence for decision support**, rather than proof of causation or a guarantee of future performance.


## 6. Ranked recommendations


Based on the analysis, the following actions are recommended in order of priority.

### 1. Use observed search signals as prioritization inputs, not causal explanations

The tested signals can be used to organize pages for further review, but they should not be interpreted as causes of CTR changes.

### 2. Keep a simple baseline when evaluating new models

The mean-CTR baseline provides an important reference point. Any future model should be required to outperform the baseline on the same held-out data before being considered useful.

### 3. Prefer client-grouped validation for cross-client evaluation

The Week-6 audit showed why validation design matters. Future experiments should keep related observations from the same client together when the goal is to estimate performance on unseen clients.

### 4. Avoid target leakage in future feature development

Features that directly contain clicks or other components of the CTR calculation should not be used to predict CTR unless the prediction setup explicitly makes those values available before the target is observed.

### 5. Collect and test additional public-safe signals

Because the current model used only a small feature set and did not outperform the baseline, future work should evaluate additional signals using the same leakage checks and grouped validation design.


## 7. Artifacts the paper embeds


### 1. Model vs Baseline Results Table

A table comparing the client-grouped Random Forest with the grouped mean-CTR baseline using the same held-out test set.

| Method                |      R² |      MAE |
| --------------------- | ------: | -------: |
| Grouped Baseline      | -0.0001 | 0.001410 |
| Grouped Random Forest | -0.0370 | 0.001392 |

### 2. R² Comparison Chart

A chart showing the R² values for the grouped baseline and Random Forest.

### 3. MAE Comparison Chart

A chart showing the MAE values for the grouped baseline and Random Forest.

### 4. Validation Split Summary

A summary of the client-grouped validation design:

* Training rows: 7,920
* Test rows: 1,980
* Training clients: 44
* Test clients: 11
* Client overlap: 0

### 5. Ranked Action Recommendations

The paper includes the ranked recommendations from the Week-7 action playbook. These recommendations translate the observed analysis into practical review and prioritization actions without claiming causation.

These artifacts are included to show the main evidence behind the research question, model comparison, validation design, and recommended actions.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.